# DynForecastLab quick start
Compare a temporal model to a learned evolution law. Run with the project .venv kernel.
The notebook runs only when you execute its cells. Short epochs are illustrative execution checks, not research rankings.

In [ ]:
from pathlib import Path
import os
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
from dynforecast.simulations import simulate
from dynforecast.experiments import run_experiment
from dynforecast.visualization import generate_report
series = simulate("oscillator", n=400, dt=0.05)
series.observations.shape, series.latent.shape

## Measurements and evolution laws
The simulator saves x and dx/dt independently from noisy/partial observations y.
Temporal models map histories to future targets; SINDy estimates a polynomial field from observed derivatives; Neural ODE learns an integrated one-step flow.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(series.time, series.observations[:,0], label="Observed position")
plt.xlabel("Time"); plt.legend();

In [ ]:
base = {"dataset": {"name": "oscillator", "n":400}, "history":16, "horizon":8,
        "targets":[0,1], "output":"results/notebook"}
runs = []
for name in ["persistence", "lstm", "sindy", "neural_ode", "hybrid"]:
    runs.append(run_experiment({**base, "model":{"name":name, "epochs":5,"width":16}}))
[(r["config"]["model"]["name"], r["status"], r.get("metrics",{}).get("rmse")) for r in runs]

In [ ]:
report = generate_report("results/notebook")
print(report / "summary.html")

## Research protocol
Use repeated seeds, fixed observed targets and predeclared hyperparameters for comparisons.
Study a clean baseline before corruption sweeps. Do not pool RMSE across different physical units.
Read ../docs/methods.md for assumptions and ../configs/experiments for longer studies.